# Hafta 4 — Toplu deneyler (VoxConverse dev)

**Grup_68 — Ses Kayıtlarında Konuşmacı Değişimlerinin Tespiti**

Bu defter hattın tamamını Colab'da çalıştırır: kurulum ve testler, örnek seslerle duman testi, eğitim/değerlendirme bölmesi, öznitelik önbelleği, AHC eşik taraması, konuşmacı değişim modeli eğitimi, E1–E5 deneyleri, grafikler ve kopyalanacak özet.

| Deney | Öznitelik | Konuşmacı ataması |
|---|---|---|
| E1 | MFCC | AHC, konuşmacı sayısı referanstan |
| E2 | MFCC | AHC, eşikle (eğitim yarısında seçilen) |
| E3 | ECAPA | AHC, konuşmacı sayısı referanstan |
| E4 | ECAPA | AHC, eşikle (eğitim yarısında seçilen) |
| E5 | ECAPA | Eğitilmiş değişim modeli + AHC (eşikle) |

**Nasıl kullanılır?**
- Çalışma zamanı: *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* (CPU ile de çalışır, daha yavaş).
- Sırayla 0 → 10 çalıştırın. Bağlantı koparsa **0. bölümü** çalıştırıp kaldığınız bölümden devam edin: 4–8. bölümler girdilerini Drive'dan okur, önbellek ve deneyler kaldığı yerden sürer.
- Uzun komutların çıktısı `Hafta4/sonuclar/gunluk/` altına da yazılır; 10. bölüm bunları özetler.

## 0. Ayarlar

Tüm yollar bu hücrededir. Başka bir şey değiştirmeniz gerekmez. Her yeniden bağlanmada önce bu hücreyi çalıştırın.

Colab'ın kendi (CUDA'lı) torch/torchaudio sürümü kullanılır. torchaudio ≥ 2.9 ile speechbrain 1.0.3 uyumsuzluğu `src/uyumluluk.py` ile giderilir (bkz. `docs/arayuz.md` 3.11).

In [ ]:
import os, sys, json, glob, time, shutil, subprocess

# --- Drive ---
KOK = "/content/drive/MyDrive/BilgisayarTasarim_Proje68"
ZIP_DRIVE = f"{KOK}/veri_kaynak/voxconverse_dev_wav.zip"   # 217 giriş: audio/ + 216 WAV (16 kHz mono)
ORNEK_SES = f"{KOK}/Hafta2/ornek_ses"                      # 5 örnek WAV
ORNEK_EDGE = f"{ORNEK_SES}/edge"                           # gürültülü, kısa, sessiz, bozuk, desteklenmeyen
H4 = f"{KOK}/Hafta4"
SONUCLAR = f"{H4}/sonuclar"
MODELLER = f"{H4}/modeller"
ONBELLEK = f"{H4}/onbellek"
BOLME = f"{H4}/bolme"
GUNLUK = f"{SONUCLAR}/gunluk"
ESIKLER = f"{MODELLER}/esikler.json"
MODEL = f"{MODELLER}/degisim_modeli.joblib"
ECAPA_ONBELLEK = f"{MODELLER}/ecapa"                       # ECAPA modeli bir kez indirilir

# --- Repo ve Colab yerel diski ---
REPO_URL = "https://github.com/HekimSefkan/Konusmaci-Tespiti.git"
DAL = "sefkan/hat"
REPO = "/content/Konusmaci-Tespiti"
ZIP_YEREL = "/content/veri/voxconverse_dev_wav.zip"         # hız için Drive'dan kopyalanır
VOX_URL = "https://github.com/joonson/voxconverse"         # master = sürüm 0.3
VOX_REPO = "/content/voxconverse"
VOX_RTTM = f"{VOX_REPO}/dev"                               # 216 referans RTTM

os.environ["KT_MODEL_ONBELLEK"] = ECAPA_ONBELLEK


def ortam_hazir():
    """Drive bağlı, repo klonlu, çalışma klasörü repo. 2-10. bölümler bununla başlar."""
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.isdir(f"{REPO}/.git"):
        raise RuntimeError("Repo yok: önce 1. bölümü (Kurulum) çalıştırın.")
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    for k in (SONUCLAR, MODELLER, ONBELLEK, BOLME, GUNLUK):
        os.makedirs(k, exist_ok=True)


def vox_hazir():
    """VoxConverse referans RTTM'leri yoksa klonlar (Colab yeniden başlayınca /content silinir)."""
    if not os.path.isdir(VOX_RTTM):
        subprocess.run(["git", "clone", "--depth", "1", VOX_URL, VOX_REPO], check=True)
    return len(glob.glob(f"{VOX_RTTM}/*.rttm"))


def zip_yolu():
    """Yerel kopya varsa onu (hızlı), yoksa Drive'daki zip'i kullanır."""
    return ZIP_YEREL if os.path.isfile(ZIP_YEREL) else ZIP_DRIVE


def calistir(komut, ad):
    """Komutu repo klasöründe çalıştırır; çıktıyı canlı gösterir, GUNLUK/<ad>.txt'ye yazar, çıkış kodunu kaydeder."""
    os.makedirs(GUNLUK, exist_ok=True)
    print("$", " ".join(komut), flush=True)
    t0 = time.time()
    with open(f"{GUNLUK}/{ad}.txt", "w", encoding="utf-8") as g:
        p = subprocess.Popen(komut, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for satir in p.stdout:
            print(satir, end="", flush=True)
            g.write(satir)
        kod = p.wait()
    kodlar_yolu = f"{GUNLUK}/kodlar.json"
    kodlar = json.load(open(kodlar_yolu, encoding="utf-8")) if os.path.isfile(kodlar_yolu) else {}
    kodlar[ad] = {"kod": kod, "sure_sn": round(time.time() - t0, 1), "tarih": time.strftime("%Y-%m-%d %H:%M:%S")}
    json.dump(kodlar, open(kodlar_yolu, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"--> çıkış kodu {kod}, süre {time.time() - t0:.0f} sn", flush=True)
    return kod


# --- GPU kontrolü ---
try:
    _g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
    GPU = _g.stdout.strip() if _g.returncode == 0 else ""
except FileNotFoundError:
    GPU = ""
print("GPU:", GPU or "YOK")
if not GPU:
    print("UYARI: GPU yok. Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU önerilir.\n"
          "CPU ile de çalışır; ECAPA önbelleği (5. bölüm) çok daha uzun sürer.")

## 1. Kurulum

Drive bağlanır, repo `sefkan/hat` dalıyla klonlanır; zaten klonluysa uzak daldaki son sürüme güncellenir (`git fetch` + `git reset --hard origin/sefkan/hat`: `git pull` ile aynı sonucu verir, Colab kopyasındaki yerel değişiklikler yüzünden takılmaz). Ardından bağımlılıklar kurulur.

**Not:** Defterin kendisi GitHub'dan açıldığı anki hâlidir; dalda defter değiştiyse Colab bağlantısından yeniden açın.

- `requirements.txt`'teki `torch`, `torchaudio` ve CPU `--extra-index-url` satırları atlanır; Colab'ın CUDA'lı torch'u kullanılır.
- Kurulum bir kez yapılır (`/content/.kt_kurulum_tamam`); hücreyi tekrar çalıştırınca atlanır.
- Önceden yüklü bir paketin (ör. numpy) sürümü değişirse hücre **çalışma zamanını yeniden başlatmanızı** ister. Yeniden başlattıktan sonra 0. ve 1. bölümü tekrar çalıştırın (kurulum atlanır).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "--branch", DAL, REPO_URL, REPO], check=True)
else:
    _onceki = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
    subprocess.run(["git", "-C", REPO, "fetch", "origin", DAL], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", DAL], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{DAL}"], check=True)  # git pull ile aynı sonuç; yerel değişiklik tutulmaz
    _sonraki = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
    print(f"Repo güncellendi: {_onceki} -> {_sonraki}" if _onceki != _sonraki else f"Repo zaten güncel: {_sonraki}")
print("Repo:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

PAKETLER = ["numpy", "scipy", "scikit-learn", "torch", "torchaudio", "speechbrain", "silero-vad", "onnxruntime",
            "librosa", "soundfile", "pyannote.metrics", "huggingface_hub", "joblib", "matplotlib", "pytest"]


def surumler():
    """Paket sürümleri, AYRI bir süreçte (bu çekirdekte yüklü eski sürümler karışmasın)."""
    k = ("import importlib.metadata as m, json\nd = {}\n"
         f"for p in {PAKETLER!r}:\n"
         "    try: d[p] = m.version(p)\n    except Exception: d[p] = None\nprint(json.dumps(d))")
    return json.loads(subprocess.run([sys.executable, "-c", k], capture_output=True, text=True).stdout)


ISARET = "/content/.kt_kurulum_tamam"
isaret = json.load(open(ISARET)) if os.path.isfile(ISARET) else None
if isaret:
    print("Kurulum daha önce yapılmış; atlanıyor.")
else:
    once = surumler()
    secilen = []
    for s in open(f"{REPO}/requirements.txt", encoding="utf-8"):
        s = s.strip()
        if not s or s.startswith("#") or s.startswith("--extra-index-url"):
            continue
        if s.startswith(("torch==", "torchaudio==")):  # Colab'ın CUDA'lı torch'u kullanılır
            continue
        secilen.append(s)
    open("/content/gereksinimler_colab.txt", "w").write("\n".join(secilen) + "\n")
    print("Kurulacak paketler:", *secilen, sep="\n  ")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/gereksinimler_colab.txt"], check=True)
    sonra = surumler()
    degisen = {p: (once[p], sonra[p]) for p in PAKETLER if once[p] != sonra[p]}
    print("\nDeğişen paketler:")
    for p, (a, b) in degisen.items():
        print(f"  {p}: {a} -> {b}")
    json.dump({"surumler": sonra}, open(ISARET, "w"))
    onceden_yuklu = [p for p, (a, b) in degisen.items() if a is not None]
    if onceden_yuklu:
        print("\n" + "!" * 78)
        print("!!  ÇALIŞMA ZAMANINI YENİDEN BAŞLATIN: Çalışma zamanı > Oturumu yeniden başlat")
        print("!!  Önceden yüklü paketler değişti:", ", ".join(onceden_yuklu))
        print("!!  Sonra 0. ve 1. bölümü tekrar çalıştırın (kurulum atlanır).")
        print("!" * 78)

print("\nSürümler:")
for p, v in surumler().items():
    print(f"  {p:<18} {v}")
# speechbrain kontrolü: önce uyumluluk yaması (torchaudio >= 2.9'da list_audio_backends yok)
r = subprocess.run([sys.executable, "-c",
                    "import torch, torchaudio; print('CUDA:', torch.cuda.is_available()); "
                    "from src.uyumluluk import speechbrain_hazirla; print('uyumluluk yaması:', speechbrain_hazirla() or 'gerekmedi'); "
                    "from speechbrain.inference.speaker import EncoderClassifier; print('speechbrain: içe aktarıldı')"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print(*r.stderr.strip().splitlines()[-5:], sep="\n")
    print("UYARI: speechbrain içe aktarılamadı; yukarıdaki hatayı 10. bölümün çıktısıyla birlikte getirin.")

## 2. Testler

`pytest` tüm testleri çalıştırır; ağır kütüphane isteyen testler (soundfile, librosa, Silero VAD, ECAPA) burada **atlanmamalıdır**. `kurulum_kontrol.py` sonunda `KURULUM: OK` yazmalıdır. ECAPA modeli ilk kez `Hafta4/modeller/ecapa` klasörüne indirilir.

In [ ]:
ortam_hazir()
kod_pytest = calistir([sys.executable, "-m", "pytest", "-q", "-rs", "-p", "no:cacheprovider", "tests"], "testler_pytest")
kod_kurulum = calistir([sys.executable, "tests/kurulum_kontrol.py"], "testler_kurulum")
print("\npytest:", "GEÇTİ" if kod_pytest == 0 else "HATA", "| kurulum_kontrol:", "GEÇTİ" if kod_kurulum == 0 else "HATA")

## 3. Duman testi

5 örnek ses ve `edge/` dosyaları için `python -m src.main` MFCC ve ECAPA ile çalıştırılır; referans RTTM'i olanlar `degerlendir` ile ölçülür.

Beklentiler: `sessiz_5sn` → boş RTTM, çıkış 0; `kisa_1sn` → çıkış 0 ve RTTM'de 0 ya da 1 konuşmacı (TG-8; VAD konuşma bulamazsa boş RTTM de başarılıdır); `bozuk_yarim` ve `desteklenmeyen_bicim` → çıkış 1; diğerleri → çıkış 0.

In [ ]:
ortam_hazir()
import pandas as pd
from src import cikti, degerlendirme

DUMAN = f"{SONUCLAR}/duman"
BEKLENEN = {"sessiz_5sn": "bos", "kisa_1sn": "tg8", "bozuk_yarim": 1, "desteklenmeyen_bicim": 1}
dosyalar = sorted(glob.glob(f"{ORNEK_SES}/*.wav")) + sorted(f for f in glob.glob(f"{ORNEK_EDGE}/*") if os.path.isfile(f))
print(len(dosyalar), "dosya:", ", ".join(os.path.basename(f) for f in dosyalar))

satirlar = []
for yontem in ("mfcc", "ecapa"):
    os.makedirs(f"{DUMAN}/{yontem}", exist_ok=True)
    for f in dosyalar:
        ad = os.path.splitext(os.path.basename(f))[0]
        hyp = f"{DUMAN}/{yontem}/{ad}.rttm"
        if os.path.exists(hyp):
            os.remove(hyp)
        t0 = time.time()
        p = subprocess.run([sys.executable, "-m", "src.main", f, "-o", hyp, "--yontem", yontem],
                           cwd=REPO, capture_output=True, text=True)
        s = {"yontem": yontem, "dosya": os.path.basename(f), "cikis": p.returncode, "sure_sn": round(time.time() - t0, 1)}
        ref = next((r for r in (f"{REPO}/data/sample/{ad}.rttm", f"{REPO}/data/sample/edge/{ad}.rttm")
                    if os.path.isfile(r)), None)
        if p.returncode == 0 and os.path.isfile(hyp):
            segler = cikti.rttm_oku(hyp)
            s["konusmaci"] = len({e for *_, e in segler})
            s["segment"] = len(segler)
            if ref:
                d = degerlendirme.degerlendir(ref, hyp)
                s.update({k: round(d[k], 4) for k in ("der", "jer", "f1")})
        elif p.returncode != 0:
            s["mesaj"] = (p.stderr.strip().splitlines() or [""])[-1][:120]
        b = BEKLENEN.get(ad)
        if b == "bos":
            ok = p.returncode == 0 and s.get("segment") == 0
        elif b == "tg8":  # TG-8: kısa dosyada 0 veya 1 konuşmacı, çıkış 0
            ok = p.returncode == 0 and s.get("konusmaci") in (0, 1)
            if ok and s.get("konusmaci") == 0:
                s["not"] = "0 konuşmacı (TG-8 karşılandı)"
        elif b == 1:
            ok = p.returncode == 1
        else:
            ok = p.returncode == 0
        s["beklenti"] = "OK" if ok else "HATA"
        satirlar.append(s)
        print(f"{yontem:<6}{ad:<45} çıkış {p.returncode}  {s['beklenti']}  {s.get('not', '')}", flush=True)

duman = pd.DataFrame(satirlar)
duman.to_csv(f"{DUMAN}/duman.csv", index=False)
display(duman)
print("Duman testi:", "GEÇTİ" if (duman.beklenti == "OK").all() else "BEKLENTİYE UYMAYAN SATIR VAR")

## 4. Veri hazırlığı

Zip Drive'dan Colab'ın yerel diskine kopyalanır (okuma hızı için; zip açılmaz). VoxConverse referans RTTM'leri klonlanır, eğitim/değerlendirme bölmesi üretilip Drive'a (`Hafta4/bolme/`) yazılır ve içeriği ekrana basılır.

In [ ]:
ortam_hazir()
import zipfile

if not os.path.isfile(ZIP_DRIVE):
    raise FileNotFoundError(ZIP_DRIVE)
os.makedirs(os.path.dirname(ZIP_YEREL), exist_ok=True)
if os.path.isfile(ZIP_YEREL) and os.path.getsize(ZIP_YEREL) == os.path.getsize(ZIP_DRIVE):
    print("Zip yerel diskte hazır.")
else:
    t0 = time.time()
    shutil.copyfile(ZIP_DRIVE, ZIP_YEREL + ".kopyalaniyor")
    os.replace(ZIP_YEREL + ".kopyalaniyor", ZIP_YEREL)
    print(f"Zip kopyalandı: {os.path.getsize(ZIP_YEREL) / 1e9:.2f} GB, {time.time() - t0:.0f} sn")
with zipfile.ZipFile(ZIP_YEREL) as z:
    wavlar = {os.path.splitext(os.path.basename(n))[0] for n in z.namelist() if n.lower().endswith(".wav")}
print(len(wavlar), "WAV (beklenen 216)")

print(vox_hazir(), "referans RTTM (beklenen 216)")
print("VoxConverse commit:", subprocess.run(["git", "-C", VOX_REPO, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())

kod_bolme = calistir([sys.executable, "-m", "src.bolme", "--rttm-dizin", VOX_RTTM, "--cikti-dizin", BOLME], "bolme")
for ad in ("egitim.txt", "degerlendirme.txt"):
    kayitlar = [s.strip() for s in open(f"{BOLME}/{ad}", encoding="utf-8") if s.strip()]
    eksik = [k for k in kayitlar if k not in wavlar]
    print(f"\n{ad}: {len(kayitlar)} kayıt" + (f"  ZİP'TE OLMAYAN: {eksik}" if eksik else ""))
    print(" ".join(kayitlar))

## 5. Öznitelik önbelleği

216 kaydın her biri için VAD bölgeleri, pencereler, MFCC (26) ve ECAPA (192) vektörleri `Hafta4/onbellek/` altına yazılır. Önbellekte olan kayıt yeniden hesaplanmaz; bağlantı koparsa hücreyi yeniden çalıştırmak yeterlidir.

In [ ]:
ortam_hazir()
kod_onbellek = calistir([sys.executable, "-m", "src.toplu", "--onbellek-doldur", "--zip", zip_yolu(),
                         "--liste", f"{BOLME}/egitim.txt", f"{BOLME}/degerlendirme.txt",
                         "--onbellek", ONBELLEK, "--cikti-dizin", SONUCLAR], "onbellek")

## 6. Eşik taraması (eğitim yarısı)

AHC kosinüs mesafesi eşiği 0,05–1,50 aralığında 0,05 adımla MFCC ve ECAPA için taranır; toplam DER'i en düşük eşik `Hafta4/modeller/esikler.json` dosyasına yazılır. E2, E4 ve E5 bu dosyayı okur. `VARSAYILAN_ESIK` değiştirilmez.

In [ ]:
ortam_hazir(); vox_hazir()
import pandas as pd
import matplotlib.pyplot as plt

kod_esik = calistir([sys.executable, "-m", "src.toplu", "--esik-tara", "--zip", zip_yolu(), "--rttm-dizin", VOX_RTTM,
                     "--liste", f"{BOLME}/egitim.txt", "--onbellek", ONBELLEK, "--esikler", ESIKLER,
                     "--cikti-dizin", SONUCLAR], "esik_tarama")
esikler = json.load(open(ESIKLER, encoding="utf-8"))
print("Seçilen eşikler:", {"mfcc": esikler.get("mfcc"), "ecapa": esikler.get("ecapa")}, "| uyarılar:", esikler.get("uyarilar"))
tarama = pd.DataFrame(esikler["tarama"])
tarama.to_csv(f"{SONUCLAR}/esik_tarama.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for y, g in tarama.groupby("yontem"):
    ax[0].plot(g.esik, g.der, marker=".", label=y)
    ax[1].plot(g.esik, g.mikro_f1, marker=".", label=y)
ax[0].set(xlabel="AHC eşiği (kosinüs mesafesi)", ylabel="DER", title="Eğitim yarısı: DER")
ax[1].set(xlabel="AHC eşiği (kosinüs mesafesi)", ylabel="mikro F1", title="Eğitim yarısı: değişim F1")
for a in ax:
    a.grid(alpha=0.3); a.legend()
fig.savefig(f"{SONUCLAR}/esik_tarama.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Değişim modeli eğitimi (eğitim yarısı)

Ardışık pencere çiftleri için özellikler (ECAPA/MFCC benzerlikleri, bağlam benzerliği, zaman boşluğu) çıkarılır; lojistik regresyon ve küçük MLP kayıt bazında gruplu 5 katlı çapraz doğrulamayla karşılaştırılır. Olasılık eşiği ve tepe aralığı ±0,5 sn toleranslı değişim F1'ine göre seçilir. Model `Hafta4/modeller/degisim_modeli.joblib` dosyasına yazılır.

In [ ]:
ortam_hazir(); vox_hazir()
import joblib
import pandas as pd

kod_model = calistir([sys.executable, "-m", "src.toplu", "--model-egit", "--zip", zip_yolu(), "--rttm-dizin", VOX_RTTM,
                      "--liste", f"{BOLME}/egitim.txt", "--onbellek", ONBELLEK, "--model", MODEL,
                      "--cikti-dizin", SONUCLAR], "model_egitimi")
paket = joblib.load(MODEL)
print(f"Seçilen model: {paket['secilen_model']} | olasılık eşiği {paket['olasilik_esigi']} | "
      f"min aralık {paket['min_aralik_sn']} sn | eğitim F1 {paket['egitim_f1']:.4f}")
print(f"Eğitim: {paket['egitim_kayit_sayisi']} kayıt, {paket['egitim_cift_sayisi']} çift, "
      f"değişim oranı {paket['degisim_orani']:.3f} | özellik sürümü {paket['ozellik_surumu']}")
for u in paket.get("uyarilar", []):
    print("UYARI:", u)
display(pd.DataFrame({ad: {"AP ort": v["ap_ort"], "AP std": v["ap_std"]} for ad, v in paket["cv"].items()}).T)

## 8. Değerlendirme yarısında E1–E5

Her deney için kayıt başına RTTM (`sonuclar/<deney>/`), kayıt CSV'si (`sonuclar/<deney>.csv`) ve toplam özet (`sonuclar/ozet.csv`) yazılır. CSV'de olan kayıtlar atlanır; yarıda kalırsa hücreyi yeniden çalıştırın.

In [ ]:
ortam_hazir(); vox_hazir()
import pandas as pd

kod_deney = {}
for deney in ("E1", "E2", "E3", "E4", "E5"):
    komut = [sys.executable, "-m", "src.toplu", "--deney", deney, "--zip", zip_yolu(), "--rttm-dizin", VOX_RTTM,
             "--liste", f"{BOLME}/degerlendirme.txt", "--cikti-dizin", SONUCLAR, "--onbellek", ONBELLEK]
    if os.path.isfile(ESIKLER):
        komut += ["--esikler", ESIKLER]
    elif deney in ("E2", "E4", "E5"):
        print(f"UYARI: {ESIKLER} yok; {deney} VARSAYILAN_ESIK kullanacak (6. bölüm).")
    if deney == "E5":
        if not os.path.isfile(MODEL):
            print("E5 atlandı: model yok (7. bölüm).")
            continue
        komut += ["--model", MODEL]
    kod_deney[deney] = calistir(komut, f"deney_{deney}")
print("Çıkış kodları:", kod_deney)
display(pd.read_csv(f"{SONUCLAR}/ozet.csv"))

## 9. Görseller

Deney başına DER / JER / değişim F1 çubuk grafiği ve bir kayıt için referans, E4 ve E5 zaman çizelgesi. PNG'ler `Hafta4/sonuclar/` altına kaydedilir. `KAYIT` boşsa E4 DER'i medyana en yakın kayıt seçilir. Her satırdaki renkler o satırın kendi etiketleridir (referans ve tahmin etiketleri eşleştirilmez).

In [ ]:
ortam_hazir(); vox_hazir()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import cikti

ozet = pd.read_csv(f"{SONUCLAR}/ozet.csv")
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(ozet))
for i, (sutun, ad) in enumerate([("der", "DER"), ("jer_ort", "JER (ort.)"), ("mikro_f1", "Değişim F1 (mikro)")]):
    ax.bar(x + (i - 1) * 0.27, ozet[sutun], 0.27, label=ad)
ax.set_xticks(x, ozet.deney)
ax.set_ylim(0, 1)
ax.set_title(f"VoxConverse dev, değerlendirme yarısı ({int(ozet.kayit_sayisi.max())} kayıt)")
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.savefig(f"{SONUCLAR}/deney_karsilastirma.png", dpi=150, bbox_inches="tight")
plt.show()

KAYIT = None
e4 = pd.read_csv(f"{SONUCLAR}/E4.csv")
if KAYIT is None:
    KAYIT = e4.loc[(e4.der - e4.der.median()).abs().idxmin(), "kayit"]
satirlar = [("Referans", f"{VOX_RTTM}/{KAYIT}.rttm")] + [(d, f"{SONUCLAR}/{d}/{KAYIT}.rttm") for d in ("E4", "E5")]
satirlar = [(ad, yol) for ad, yol in satirlar if os.path.isfile(yol)]
fig, ax = plt.subplots(figsize=(14, 0.9 * len(satirlar) + 1))
for j, (ad, yol) in enumerate(satirlar):
    segler = cikti.rttm_oku(yol)
    renk = {e: plt.cm.tab20(i % 20) for i, e in enumerate(sorted({e for *_, e in segler}))}
    y = len(satirlar) - 1 - j
    for a, b, e in segler:
        ax.broken_barh([(a, b - a)], (y - 0.4, 0.8), color=renk[e])
ax.set_yticks(range(len(satirlar)), [ad for ad, _ in reversed(satirlar)])
ax.set_xlabel("zaman (sn)")
ders = {d: pd.read_csv(f"{SONUCLAR}/{d}.csv").set_index("kayit").der.get(KAYIT)
        for d in ("E4", "E5") if os.path.isfile(f"{SONUCLAR}/{d}.csv")}
ax.set_title(f"{KAYIT}  " + "  ".join(f"{d} DER={v:.3f}" for d, v in ders.items() if v is not None))
fig.savefig(f"{SONUCLAR}/zaman_cizelgesi_{KAYIT}.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Kopyalanacak özet

Aşağıdaki çıktının tamamını kopyalayıp geri getirin: özet tablosu, eşikler, model seçimi ve çapraz doğrulama, bölme dosyaları, çıkış kodları, duman testi ve hata varsa ilgili günlüğün son 30 satırı.

In [ ]:
ortam_hazir()
import pandas as pd


def baslik(s):
    print("\n" + "=" * 72 + f"\n{s}\n" + "=" * 72)


def son_satirlar(yol, n=30):
    if os.path.isfile(yol):
        print(f"--- {os.path.relpath(yol, KOK)} (son {n} satır) ---")
        print("".join(open(yol, encoding="utf-8", errors="replace").readlines()[-n:]).rstrip())


baslik("ORTAM")
print("GPU:", GPU or "YOK")
print("Repo:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
if os.path.isfile("/content/.kt_kurulum_tamam"):
    s = json.load(open("/content/.kt_kurulum_tamam"))
    print(", ".join(f"{p}={v}" for p, v in s["surumler"].items()))

baslik("OZET.CSV (değerlendirme yarısı)")
p = f"{SONUCLAR}/ozet.csv"
print(pd.read_csv(p).to_string(index=False) if os.path.isfile(p) else "yok")

baslik("EŞİKLER (eğitim yarısı)")
if os.path.isfile(ESIKLER):
    e = json.load(open(ESIKLER, encoding="utf-8"))
    print("mfcc:", e.get("mfcc"), "| ecapa:", e.get("ecapa"), "| kayıt:", e.get("kayit_sayisi"),
          "| hatalı:", e.get("hatali"), "| uyarılar:", e.get("uyarilar"))
    t = pd.DataFrame(e["tarama"])
    print(t.pivot(index="esik", columns="yontem", values=["der", "mikro_f1"]).round(4).to_string())
else:
    print("yok")

baslik("DEĞİŞİM MODELİ")
if os.path.isfile(MODEL):
    import joblib
    pk = joblib.load(MODEL)
    print(f"seçilen: {pk['secilen_model']} | olasılık eşiği: {pk['olasilik_esigi']} | min aralık: {pk['min_aralik_sn']} sn"
          f" | eğitim F1: {pk['egitim_f1']:.4f} | kayıt: {pk['egitim_kayit_sayisi']} | çift: {pk['egitim_cift_sayisi']}"
          f" | değişim oranı: {pk['degisim_orani']:.3f} | sklearn {pk['sklearn_surumu']} | {pk['egitim_tarihi']}")
    print("uyarılar:", pk.get("uyarilar", []))
    for ad, v in pk["cv"].items():
        print(f"  {ad:<22} AP {v['ap_ort']:.4f} ± {v['ap_std']:.4f}")
else:
    print("yok")

baslik("BÖLME")
for ad in ("egitim.txt", "degerlendirme.txt"):
    p = f"{BOLME}/{ad}"
    if os.path.isfile(p):
        k = [s.strip() for s in open(p, encoding="utf-8") if s.strip()]
        print(f"{ad} ({len(k)}): " + " ".join(k))

baslik("DUMAN TESTİ")
p = f"{SONUCLAR}/duman/duman.csv"
print(pd.read_csv(p).to_string(index=False) if os.path.isfile(p) else "yok")

baslik("ÇIKIŞ KODLARI")
kodlar = json.load(open(f"{GUNLUK}/kodlar.json", encoding="utf-8")) if os.path.isfile(f"{GUNLUK}/kodlar.json") else {}
for ad, v in kodlar.items():
    print(f"  {ad:<18} kod {v['kod']}  {v['sure_sn']:>8.0f} sn  {v['tarih']}")
son_satirlar(f"{GUNLUK}/testler_pytest.txt", 12)
for ad, v in kodlar.items():
    if v["kod"] != 0:
        son_satirlar(f"{GUNLUK}/{ad}.txt", 30)
son_satirlar(f"{SONUCLAR}/hata_gunlugu.txt", 30)